# Reproduce the Swift codec training experiments

This notebook runs the original training experiments for the layered neural video codec.

The experiments use the main training script `codec/src/train.py`.
The script trains one model for each temporal-hierarchy level.

The notebook uses the data in `data/`.
It ignores the masks and the edge maps.

## Result

The notebook writes three trained models under `codec/src/experiments/`.
Each model contains an encoder, a binarizer, a decoder, a layered decoder, and a U-Net.

## Before you start

### 1. Select the environment

Run this command in a terminal:

```bash
conda activate semswift_env
```

### 2. Register the kernel

Run these commands in the same terminal:

```bash
python -m ipykernel install --sys-prefix --name semswift_env --display-name "Python (semswift_env)"
python -m jupyter kernelspec list
```

The second command must list `semswift_env`.
`--sys-prefix` writes the kernel into `semswift_env`.

### 3. Start JupyterLab

Run these commands:

```bash
cd /nfs/home/olidiz18/git/S-SWIFT/codec/src
python -m jupyter lab
```

Start JupyterLab after you register the kernel.
JupyterLab reads the kernel list at server start.

### 4. Select the kernel

Open the notebook.
Select `Kernel > Change Kernel > Python (semswift_env)`.
If JupyterLab does not show the kernel, stop the server with `Ctrl-C` and start it again.
Then reload the browser page.

The `python3` kernel in `semswift_env` also uses the correct packages.
Use `Python (semswift_env)` when it is available.

### 5. Run the training cells

Run the notebook cells from top to bottom.
Run one training cell at a time.
Each training cell uses both GPUs.
A full training run can take many hours.
Set `QUICK_TEST = True` in the configuration cell for a short check.

### VS Code

If you use the VS Code notebook editor, select the Jupyter kernel `Python (semswift_env)`.
If the kernel is not listed, select the Python environment `semswift_env` instead.
Reload the VS Code window after you register the kernel.

## Scope

- This notebook reproduces the three hierarchy runs from `codec/src/train.sh`.
- It runs `train.py`. It does not run the adversarial variant `train_adv.py`.
- It does not use `data/train_masks`, `data/train_edges`, `data/eval_masks`, or `data/eval_edges`.
- It writes a 352x288 frame cache. The motion vectors in `data/train_mv` and `data/eval_mv` are 352x288. This size matches the original experiments.
- The notebook calls the training script as a subprocess. The training logic stays in `train.py`.
- The `data/` tree is a small sample. The full paper run used a larger training set.

In [1]:
from pathlib import Path
from datetime import datetime
import os
import signal
import subprocess
import sys

import cv2

PREFERRED_GPU_IDS = [0, 1]
AUTO_SELECT_GPUS = True
MIN_FREE_GPU_MEMORY_MB = 12288


def find_repo_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "codec" / "src" / "train.py").is_file():
            return candidate
    raise FileNotFoundError(
        "Cannot find the repository root. Start Jupyter in codec/src or in the repository root."
    )


def query_free_gpu_memory():
    command = [
        "nvidia-smi",
        "--query-gpu=index,memory.free",
        "--format=csv,noheader,nounits",
    ]
    result = subprocess.run(command, capture_output=True, text=True, check=True)
    free_memory = {}
    for line in result.stdout.strip().splitlines():
        index, memory = [part.strip() for part in line.split(",")]
        free_memory[int(index)] = int(memory)
    return free_memory


REPO_ROOT = find_repo_root()
SRC_DIR = REPO_ROOT / "codec" / "src"
DATA_DIR = REPO_ROOT / "data"
TRAIN_MV_DIR = DATA_DIR / "train_mv"
EVAL_MV_DIR = DATA_DIR / "eval_mv"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

if AUTO_SELECT_GPUS:
    try:
        free_memory = query_free_gpu_memory()
    except (FileNotFoundError, subprocess.CalledProcessError):
        print("Warning: Cannot read GPU memory from nvidia-smi.")
        free_memory = {
            index: MIN_FREE_GPU_MEMORY_MB for index in PREFERRED_GPU_IDS
        }

    SELECTED_PHYSICAL_GPU_IDS = [
        index for index in PREFERRED_GPU_IDS
        if free_memory.get(index, 0) >= MIN_FREE_GPU_MEMORY_MB
    ]
    if not SELECTED_PHYSICAL_GPU_IDS:
        print("Warning: No preferred GPU has enough free memory.")
        print("Free memory (MiB):", free_memory)
        SELECTED_PHYSICAL_GPU_IDS = list(PREFERRED_GPU_IDS)
else:
    SELECTED_PHYSICAL_GPU_IDS = list(PREFERRED_GPU_IDS)

os.environ["CUDA_VISIBLE_DEVICES"] = ",".join(
    str(index) for index in SELECTED_PHYSICAL_GPU_IDS
)
GPU_IDS = list(range(len(SELECTED_PHYSICAL_GPU_IDS)))
GPU_ARG = ",".join(str(index) for index in GPU_IDS)

if "torch" in sys.modules:
    print("Warning: Restart the kernel before you rerun this cell. CUDA was already initialized.")

import torch

print("Kernel Python:", sys.executable)
if "semswift_env" not in sys.executable:
    print("Warning: This kernel does not use semswift_env. Select the Python (semswift_env) kernel.")

print("Repository root:", REPO_ROOT)
print("Source directory:", SRC_DIR)
print("Data directory:", DATA_DIR)
print("Physical GPU IDs:", SELECTED_PHYSICAL_GPU_IDS)
print("CUDA_VISIBLE_DEVICES:", os.environ["CUDA_VISIBLE_DEVICES"])
print("Logical GPU IDs:", GPU_IDS)

Kernel Python: /nfs/home/olidiz18/conda_envs/semswift_env/bin/python
Repository root: /nfs/home/olidiz18/git/S-SWIFT
Source directory: /nfs/home/olidiz18/git/S-SWIFT/codec/src
Data directory: /nfs/home/olidiz18/git/S-SWIFT/data
Physical GPU IDs: [0, 1]
CUDA_VISIBLE_DEVICES: 0,1
Logical GPU IDs: [0, 1]


## Configuration

The configuration repeats the settings in `train.sh`.
`QUICK_TEST` changes only the iteration counts. It does not change the model.
`NUM_WORKERS` sets the number of DataLoader processes.
`NUM_CROPS` sets how many crops each frame contributes.
`EVAL_BATCH_SIZE` sets the evaluation batch size.
`EARLY_STOP_PATIENCE` stops training after that many evaluations without PSNR improvement. Use `0` to disable it.
`RESUME_FROM` continues a hierarchy run from a saved checkpoint.
Set a value to `None` to start that hierarchy from scratch.

In [2]:
HIERARCHIES = {
    0: {"distance1": 6, "distance2": 6, "bits": 16, "encoder_fuse_level": 1, "decoder_fuse_level": 1},
    1: {"distance1": 3, "distance2": 3, "bits": 16, "encoder_fuse_level": 2, "decoder_fuse_level": 3},
    2: {"distance1": 1, "distance2": 2, "bits": 8, "encoder_fuse_level": 1, "decoder_fuse_level": 1},
}

QUICK_TEST = False
MAX_TRAIN_ITERS = 20 if QUICK_TEST else 20000
CHECKPOINT_ITERS = 10 if QUICK_TEST else 5000
EVAL_ITERS = 10 if QUICK_TEST else 4000
SCHEDULE = ",".join(
    str(int(MAX_TRAIN_ITERS * fraction))
    for fraction in [0.5, 0.6, 0.7, 0.8, 0.9]
)

BATCH_SIZE = 8
EVAL_BATCH_SIZE = 8
NUM_CROPS = 2
NUM_WORKERS = 4

RESUME_FROM = {0: None, 1: None, 2: None}
EARLY_STOP_PATIENCE = 0

SAVE_CODES = True
SAVE_OUT_IMG = True
TENSORBOARD = True
TENSORBOARD_PORT = 6006

FRAME_WIDTH = 352
FRAME_HEIGHT = 288
PREPARED_ROOT = DATA_DIR / f"prepared_{FRAME_WIDTH}x{FRAME_HEIGHT}"

EXPERIMENT_ROOT = SRC_DIR / "experiments"


def model_dir(level):
    return EXPERIMENT_ROOT / f"hierarchy_{level}" / "model"


def output_dir(level):
    return EXPERIMENT_ROOT / f"hierarchy_{level}" / "output"


def log_path(level):
    return EXPERIMENT_ROOT / f"hierarchy_{level}" / "train.log"


def tensorboard_dir(level):
    base = EXPERIMENT_ROOT / f"hierarchy_{level}" / "tensorboard"
    resume_iter = RESUME_FROM.get(level)
    if resume_iter:
        return base / f"resume_{resume_iter}"
    return base


def checkpoint_prefix(level):
    return f"hierarchy{level}"


for level in HIERARCHIES:
    model_dir(level).mkdir(parents=True, exist_ok=True)
    output_dir(level).mkdir(parents=True, exist_ok=True)

print("Experiment root:", EXPERIMENT_ROOT)
print("Prepared data root:", PREPARED_ROOT)
print("Selected physical GPUs:", SELECTED_PHYSICAL_GPU_IDS)
print("Logical GPU IDs:", GPU_IDS)
print("CUDA_VISIBLE_DEVICES:", os.environ["CUDA_VISIBLE_DEVICES"])
print("Save codes:", SAVE_CODES)
print("Save output images:", SAVE_OUT_IMG)
print("TensorBoard:", TENSORBOARD)
print("TensorBoard port:", TENSORBOARD_PORT)
print("DataLoader workers:", NUM_WORKERS)
print("Resume from:", RESUME_FROM)
print("Num crops:", NUM_CROPS)
print("Eval batch size:", EVAL_BATCH_SIZE)
print("Early stop patience:", EARLY_STOP_PATIENCE)

Experiment root: /nfs/home/olidiz18/git/S-SWIFT/codec/src/experiments
Prepared data root: /nfs/home/olidiz18/git/S-SWIFT/data/prepared_352x288
Selected physical GPUs: [0, 1]
Logical GPU IDs: [0, 1]
CUDA_VISIBLE_DEVICES: 0,1
Save codes: True
Save output images: True
TensorBoard: True
TensorBoard port: 6006
DataLoader workers: 4
Resume from: {0: None, 1: None, 2: None}
Num crops: 2
Eval batch size: 8
Early stop patience: 0


## GPU check

The training script uses `torch.nn.DataParallel` with `--gpus 0,1`.
The notebook checks the GPU count before each run.

In [3]:
print("PyTorch version:", torch.__version__)
print("PyTorch CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("Visible GPU count:", torch.cuda.device_count())
print("Physical GPU IDs:", SELECTED_PHYSICAL_GPU_IDS)
print("Logical GPU IDs:", GPU_IDS)

if torch.cuda.is_available():
    for logical in GPU_IDS:
        free, total = torch.cuda.mem_get_info(logical)
        print(
            f"Logical GPU {logical}: {torch.cuda.get_device_name(logical)} "
            f"free={free / 1024 ** 3:.1f} GiB total={total / 1024 ** 3:.1f} GiB"
        )
else:
    print(
        "Warning: CUDA is not available. Training will stop. "
        "Check the NVIDIA driver and the PyTorch CUDA build."
    )


def require_gpus():
    if len(GPU_IDS) == 0:
        raise RuntimeError(
            "No GPU is selected. Set PREFERRED_GPU_IDS or wait for a free GPU."
        )
    if not torch.cuda.is_available() or torch.cuda.device_count() < len(GPU_IDS):
        raise RuntimeError(
            f"Training needs {len(GPU_IDS)} visible GPUs. "
            f"Found {torch.cuda.device_count()}. "
            "Check the NVIDIA driver and the PyTorch CUDA build."
        )
    for logical in GPU_IDS:
        free, _ = torch.cuda.mem_get_info(logical)
        free_mb = free // (1024 ** 2)
        if free_mb < MIN_FREE_GPU_MEMORY_MB:
            raise RuntimeError(
                f"Logical GPU {logical} has {free_mb} MiB free. "
                f"Training needs at least {MIN_FREE_GPU_MEMORY_MB} MiB. "
                "Wait for a free GPU or change PREFERRED_GPU_IDS."
            )

PyTorch version: 2.14.1+cu126
PyTorch CUDA build: 12.6
CUDA available: True
Visible GPU count: 2
Physical GPU IDs: [0, 1]
Logical GPU IDs: [0, 1]
Logical GPU 0: NVIDIA RTX 4000 Ada Generation free=19.4 GiB total=19.5 GiB
Logical GPU 1: NVIDIA RTX 4000 Ada Generation free=19.4 GiB total=19.5 GiB


## Prepare the training data

The motion vectors are 352x288.
The source frames have different sizes.
The next cell writes a 352x288 frame cache under `data/`.
The cell does not copy the motion vectors.
The cell skips frames that already exist in the cache.

In [4]:
def prepare_frames(source_dir, target_dir):
    target_dir.mkdir(parents=True, exist_ok=True)
    written = 0

    for clip_dir in sorted(path for path in source_dir.iterdir() if path.is_dir()):
        target_clip = target_dir / clip_dir.name
        target_clip.mkdir(parents=True, exist_ok=True)

        for frame_path in sorted(clip_dir.glob("frame_*.png")):
            target_path = target_clip / frame_path.name
            if target_path.exists():
                written += 1
                continue

            image = cv2.imread(str(frame_path))
            if image is None:
                raise RuntimeError(f"Cannot read {frame_path}")

            if image.shape[1] != FRAME_WIDTH or image.shape[0] != FRAME_HEIGHT:
                image = cv2.resize(
                    image,
                    (FRAME_WIDTH, FRAME_HEIGHT),
                    interpolation=cv2.INTER_AREA,
                )

            if not cv2.imwrite(str(target_path), image):
                raise RuntimeError(f"Cannot write {target_path}")

            written += 1

    return written


train_count = prepare_frames(DATA_DIR / "train", PREPARED_ROOT / "train")
eval_count = prepare_frames(DATA_DIR / "eval", PREPARED_ROOT / "eval")

print(f"Training frames: {train_count}")
print(f"Evaluation frames: {eval_count}")
print("Prepared data:", PREPARED_ROOT)

Training frames: 3279
Evaluation frames: 300
Prepared data: /nfs/home/olidiz18/git/S-SWIFT/data/prepared_352x288


## Check the data

The next cell builds the dataset objects for each hierarchy level.
The count depends on the hierarchy positions and on complete frame groups.

In [5]:
from dataset import ImageFolder
from train_options import parser


def base_arguments(level):
    config = HIERARCHIES[level]
    return [
        "--train", str(PREPARED_ROOT / "train"),
        "--eval", str(PREPARED_ROOT / "eval"),
        "--train-mv", str(TRAIN_MV_DIR),
        "--eval-mv", str(EVAL_MV_DIR),
        "--encoder-fuse-level", str(config["encoder_fuse_level"]),
        "--decoder-fuse-level", str(config["decoder_fuse_level"]),
        "--v-compress", "--warp", "--stack", "--fuse-encoder",
        "--bits", str(config["bits"]),
        "--distance1", str(config["distance1"]),
        "--distance2", str(config["distance2"]),
        "--num-workers", str(NUM_WORKERS),
    ]


def dataset_arguments(level):
    return parser.parse_args(
        base_arguments(level)
        + [
            "--batch-size", str(BATCH_SIZE),
            "--eval-batch-size", str(EVAL_BATCH_SIZE),
            "--num-crops", str(NUM_CROPS),
            "--gpus", GPU_ARG,
        ]
    )


for level in HIERARCHIES:
    arguments = dataset_arguments(level)
    train_dataset = ImageFolder(
        is_train=True,
        root=str(PREPARED_ROOT / "train"),
        mv_dir=str(TRAIN_MV_DIR),
        args=arguments,
    )
    eval_dataset = ImageFolder(
        is_train=False,
        root=str(PREPARED_ROOT / "eval"),
        mv_dir=str(EVAL_MV_DIR),
        args=arguments,
    )
    print(
        f"Hierarchy {level}: {len(train_dataset)} training frames, "
        f"{len(eval_dataset)} evaluation frames."
    )

268 images loaded.
	distance=6/6
24 images loaded.
	distance=6/6
Hierarchy 0: 268 training frames, 24 evaluation frames.
541 images loaded.
	distance=3/3
49 images loaded.
	distance=3/3
Hierarchy 1: 541 training frames, 49 evaluation frames.
2174 images loaded.
	distance=1/2
198 images loaded.
	distance=1/2
Hierarchy 2: 2174 training frames, 198 evaluation frames.


## Multi-GPU training

Each training cell starts `train.py` with `torchrun` and `DistributedDataParallel`.
The command launches one process per selected GPU.
The log file is `codec/src/experiments/hierarchy_<level>/train.log`.

Run one cell at a time.

In [6]:
def training_command(level):
    command = [
        sys.executable, "-m", "torch.distributed.run",
        "--nproc_per_node", str(len(GPU_IDS)),
        str(SRC_DIR / "train.py"),
        *base_arguments(level),
        "--max-train-iters", str(MAX_TRAIN_ITERS),
        "--checkpoint-iters", str(CHECKPOINT_ITERS),
        "--eval-iters", str(EVAL_ITERS),
        "--model-dir", str(model_dir(level)),
        "--out-dir", str(output_dir(level)),
        "--save-model-name", checkpoint_prefix(level),
        "--batch-size", str(BATCH_SIZE),
        "--eval-batch-size", str(EVAL_BATCH_SIZE),
        "--num-crops", str(NUM_CROPS),
        "--mp-context", "spawn",
        "--schedule", SCHEDULE,
    ]
    resume_iter = RESUME_FROM.get(level)
    if resume_iter:
        command.extend([
            "--load-model-name", checkpoint_prefix(level),
            "--load-iter", str(resume_iter),
        ])
    if EARLY_STOP_PATIENCE > 0:
        command.extend(["--early-stop", str(EARLY_STOP_PATIENCE)])
    if SAVE_CODES:
        command.append("--save-codes")
    if SAVE_OUT_IMG:
        command.append("--save-out-img")
    if TENSORBOARD:
        command.extend(["--tensorboard-dir", str(tensorboard_dir(level))])
    return command


def run_training(level):
    require_gpus()
    command = training_command(level)
    log_file = log_path(level)
    log_file.parent.mkdir(parents=True, exist_ok=True)

    print("Command:")
    print(" ".join(str(part) for part in command))
    print("Physical GPU IDs:", SELECTED_PHYSICAL_GPU_IDS)
    print("Logical GPU IDs:", GPU_ARG)
    print("Log file:", log_file)

    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"

    process = subprocess.Popen(
        command,
        cwd=SRC_DIR,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    if log_file.exists():
        backup = log_file.with_name(
            f"train_{datetime.now():%Y%m%d_%H%M%S}.log"
        )
        log_file.replace(backup)

    try:
        with log_file.open("w") as log:
            for line in process.stdout:
                print(line, end="")
                log.write(line)
        return_code = process.wait()
    except KeyboardInterrupt:
        process.terminate()
        process.wait()
        raise

    if return_code != 0:
        if return_code < 0:
            signal_number = -return_code
            try:
                signal_name = signal.Signals(signal_number).name
            except ValueError:
                signal_name = str(signal_number)
            raise RuntimeError(
                f"Training was killed by signal {signal_name} ({return_code}). "
                f"Read {log_file}. SIGKILL usually means the OS out-of-memory "
                "killer or an administrator stopped the process."
            )
        raise RuntimeError(f"Training stopped with code {return_code}. Read {log_file}.")

    print("Training finished.")

## TensorBoard

The training script writes TensorBoard events to `codec/src/experiments/hierarchy_<level>/tensorboard/`.
TensorBoard shows the loss curves, the learning rate, the evaluation metrics, and one sample reconstruction at each evaluation.

Run the next cell after the first training cell starts.
TensorBoard reads new events automatically.
The notebook magic starts TensorBoard on the server's localhost.
If the panel is blank, forward port 6006 and open `http://localhost:6006` directly.

If the notebook magic does not start, run this command in a terminal:

```bash
cd /nfs/home/olidiz18/git/S-SWIFT
tensorboard --logdir codec/src/experiments --port 6006 --bind_all
```

For a remote server, forward the port:

```bash
ssh -L 6006:localhost:6006 <user>@<server>
```

Then open `http://localhost:6006`.

In [7]:
try:
    ipython = get_ipython()
except NameError:
    ipython = None

if ipython is None:
    print("Start TensorBoard in a terminal:")
    print(f"tensorboard --logdir {EXPERIMENT_ROOT} --port {TENSORBOARD_PORT} --bind_all")
else:
    ipython.run_line_magic("load_ext", "tensorboard")
    ipython.run_line_magic(
        "tensorboard",
        f"--logdir {EXPERIMENT_ROOT} --port {TENSORBOARD_PORT}",
    )

Reusing TensorBoard on port 6006 (pid 2609172), started 7:08:47 ago. (Use '!kill 2609172' to kill it.)

### Train hierarchy level 0

Distances 6 and 6. Bits 16.
This run uses both GPUs.

In [ ]:
run_training(0)

Command:
/nfs/home/olidiz18/conda_envs/semswift_env/bin/python -m torch.distributed.run --nproc_per_node 2 /nfs/home/olidiz18/git/S-SWIFT/codec/src/train.py --train /nfs/home/olidiz18/git/S-SWIFT/data/prepared_352x288/train --eval /nfs/home/olidiz18/git/S-SWIFT/data/prepared_352x288/eval --train-mv /nfs/home/olidiz18/git/S-SWIFT/data/train_mv --eval-mv /nfs/home/olidiz18/git/S-SWIFT/data/eval_mv --encoder-fuse-level 1 --decoder-fuse-level 1 --v-compress --warp --stack --fuse-encoder --bits 16 --distance1 6 --distance2 6 --num-workers 4 --max-train-iters 20000 --checkpoint-iters 5000 --eval-iters 4000 --model-dir /nfs/home/olidiz18/git/S-SWIFT/codec/src/experiments/hierarchy_0/model --out-dir /nfs/home/olidiz18/git/S-SWIFT/codec/src/experiments/hierarchy_0/output --save-model-name hierarchy0 --batch-size 8 --eval-batch-size 8 --num-crops 2 --mp-context spawn --schedule 10000,12000,14000,16000,18000 --save-codes --save-out-img --tensorboard-dir /nfs/home/olidiz18/git/S-SWIFT/codec/src/ex

### Train hierarchy level 1

Distances 3 and 3. Bits 16.
This run uses both GPUs.

In [ ]:
run_training(1)

### Train hierarchy level 2

Distances 1 and 2. Bits 8.
This run uses both GPUs.

In [ ]:
run_training(2)

## Evaluation

The training script evaluates the model during training.
The next cells load a saved checkpoint and run the evaluation again.
The evaluation uses the `silent` set in `data/eval`.

In [ ]:
from dataset import get_loader
from evaluate import run_eval
from util import get_models, set_eval

import network

CHECKPOINT_NAMES = ["encoder", "binarizer", "decoder", "d2", "unet"]


def evaluation_arguments(level):
    arguments = base_arguments(level) + [
        "--model-dir", str(model_dir(level)),
        "--out-dir", str(output_dir(level)),
        "--save-model-name", checkpoint_prefix(level),
        "--batch-size", str(BATCH_SIZE),
        "--eval-batch-size", str(EVAL_BATCH_SIZE),
        "--num-crops", str(NUM_CROPS),
        "--gpus", GPU_ARG,
    ]
    if SAVE_CODES:
        arguments.append("--save-codes")
    if SAVE_OUT_IMG:
        arguments.append("--save-out-img")
    return parser.parse_args(arguments)


def build_evaluation_model(level):
    args = evaluation_arguments(level)
    encoder, binarizer, decoder, unet = get_models(
        args=args,
        v_compress=args.v_compress,
        bits=args.bits,
        encoder_fuse_level=args.encoder_fuse_level,
        decoder_fuse_level=args.decoder_fuse_level,
    )
    d2 = network.DecoderCell2(
        v_compress=args.v_compress,
        shrink=args.shrink,
        bits=args.bits,
        fuse_level=args.decoder_fuse_level,
    ).cuda()

    nets = [encoder, binarizer, decoder, d2]
    if unet is not None:
        nets.append(unet)
    if len(GPU_IDS) > 1:
        nets = [torch.nn.DataParallel(net, device_ids=GPU_IDS) for net in nets]
    return args, nets


def latest_iteration(level):
    prefix = checkpoint_prefix(level)
    paths = [
        p for p in model_dir(level).glob(f"{prefix}_encoder_*.pth")
        if p.stem.rsplit("_", 1)[-1].isdigit()
    ]
    if not paths:
        raise FileNotFoundError(f"No checkpoint found in {model_dir(level)}")
    return int(paths[-1].stem.rsplit("_", 1)[-1])


def load_checkpoint(level, iteration, nets):
    prefix = checkpoint_prefix(level)
    for name, net in zip(CHECKPOINT_NAMES, nets):
        path = model_dir(level) / f"{prefix}_{name}_{iteration:08d}.pth"
        if not path.exists():
            raise FileNotFoundError(path)
        state = torch.load(path, map_location="cuda:0")
        target = net.module if isinstance(net, torch.nn.DataParallel) else net
        try:
            target.load_state_dict(state)
        except RuntimeError:
            state = {key.replace("module.", ""): value for key, value in state.items()}
            target.load_state_dict(state)


def evaluate_checkpoint(level, iteration=None):
    require_gpus()
    if iteration is None:
        iteration = latest_iteration(level)

    args, nets = build_evaluation_model(level)
    load_checkpoint(level, iteration, nets)
    set_eval(nets)

    loader = get_loader(False, str(PREPARED_ROOT / "eval"), str(EVAL_MV_DIR), args)
    result = run_eval(
        nets,
        loader,
        args,
        output_suffix=f"hierarchy{level}_iter{iteration}",
    )
    loss, msssim, psnr, psnr_ee1, psnr_ee2, psnr_ee3, psnr_ee4 = result

    print(f"Hierarchy {level}, iteration {iteration}")
    print(
        f"{'Step':>4} {'Loss':>10} {'MS-SSIM':>10} {'PSNR':>10} "
        f"{'EE1':>10} {'EE2':>10} {'EE3':>10} {'EE4':>10}"
    )
    for step in range(len(loss)):
        print(
            f"{step + 1:>4} {loss[step]:>10.5f} {msssim[step]:>10.5f} "
            f"{psnr[step]:>10.5f} {psnr_ee1[step]:>10.5f} "
            f"{psnr_ee2[step]:>10.5f} {psnr_ee3[step]:>10.5f} "
            f"{psnr_ee4[step]:>10.5f}"
        )

    return {
        "loss": loss,
        "msssim": msssim,
        "psnr": psnr,
        "psnr_ee1": psnr_ee1,
        "psnr_ee2": psnr_ee2,
        "psnr_ee3": psnr_ee3,
        "psnr_ee4": psnr_ee4,
    }

### Run evaluation

Set `EVAL_LEVEL` to a hierarchy level.
Set `EVAL_ITERATION` to a checkpoint iteration.
Use `None` for the latest checkpoint.

In [ ]:
EVAL_LEVEL = 2
EVAL_ITERATION = None

results = evaluate_checkpoint(EVAL_LEVEL, EVAL_ITERATION)

## Artifacts

- Models: `codec/src/experiments/hierarchy_<level>/model/`
- Logs: `codec/src/experiments/hierarchy_<level>/train.log`
- TensorBoard events: `codec/src/experiments/hierarchy_<level>/tensorboard/`
- Codes: `codec/src/experiments/hierarchy_<level>/output/iter<iteration>/codes/`
- Images: `codec/src/experiments/hierarchy_<level>/output/iter<iteration>/images/`
- Prepared frames: `data/prepared_352x288/`

Do not commit the prepared cache or the experiment outputs.

## Notes

- The masks and the edge maps are not used.
- The training script writes codes and images only when `SAVE_CODES` or `SAVE_OUT_IMG` is `True`.
- The training script saves a `best` checkpoint when evaluation PSNR improves.
- `NUM_CROPS = 4` amortizes the U-Net but uses more GPU memory and slows `DataParallel`. The default is `2`.
- Output images can use much disk space. Set `SAVE_OUT_IMG = False` to disable them.
- The training data and the evaluation data use the same 352x288 size as the motion vectors.
- The notebook uses all preferred GPUs that have enough free memory.
- `DistributedDataParallel` runs one process per selected GPU.
- The notebook skips a GPU with less than 12 GiB free.
- A multi-GPU checkpoint has no `module.` prefix. The source code removes the wrapper before it saves a checkpoint.

## Troubleshooting

- If `torch.cuda.is_available()` is `False`, check the NVIDIA driver and the PyTorch CUDA build for `semswift_env`.
- The server driver supports CUDA 12.9. Use a PyTorch CUDA 12.6 build:
  `python -m pip install torch==2.14.1+cu126 torchvision==0.29.1+cu126 --index-url https://download.pytorch.org/whl/cu126`
- Restart the notebook kernel after you install a new PyTorch build.
- If TensorBoard reports `No module named 'pkg_resources'`, run `python -m pip install "setuptools<81"`.
- If a GPU has less than 12 GiB free, wait for it or set `PREFERRED_GPU_IDS`.
- Set `AUTO_SELECT_GPUS = False` to use `PREFERRED_GPU_IDS` without the free-memory filter.
- If the data check shows zero frames, check the clip folders in `data/train` and `data/eval`.
- If a training cell stops, read the log file for the last error.
- If training stops with code -9, the operating system killed the process. Check host memory and set `RESUME_FROM` to the last checkpoint.
- The notebook uses persistent DataLoader workers to prevent worker restarts from leaking host memory.
- Set `NUM_WORKERS = 0` for a short diagnostic run if you suspect a DataLoader memory leak.

## Source compatibility

The notebook uses the current Python environment.
The repository source has small compatibility changes:

- `dataset.py` reads clip subfolders, `.png` motion vectors, and resized motion vectors.
- `util.py` and `evaluate.py` do not use the removed `scipy.misc` module.
- `train.py` uses `DistributedDataParallel` when more than one GPU is selected.